# Evaluate LAMPS trên D1
Yêu cầu trên Drive:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- `NT230/data/d1/test.jsonl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, sys
DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'

!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

os.makedirs('/content/saved_models/checkpoint-best-acc', exist_ok=True)
shutil.copy(f'{DRIVE_D1}/saved_models/checkpoint-best-acc/model.bin',
            '/content/saved_models/checkpoint-best-acc/model.bin')
shutil.copy(f'{DRIVE_D1}/test.jsonl', '/content/test.jsonl')
print('✅ model.bin:', round(os.path.getsize('/content/saved_models/checkpoint-best-acc/model.bin')/1e6), 'MB')
print('✅ test.jsonl:', sum(1 for _ in open('/content/test.jsonl')), 'records')

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm

In [ ]:
import json
from pathlib import Path
from lamps.models.codebert import CodeBERTClassifier
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

clf = CodeBERTClassifier(
    checkpoint='/content/saved_models/checkpoint-best-acc/model.bin',
    base_model='microsoft/codebert-base',
    block_size=400,
)

records = list(read_jsonl(Path('/content/test.jsonl')))
codes   = [r['func'] for r in records]
y_true  = [r['target'] for r in records]
print(f'Test samples: {len(records)}')

preds  = clf.predict_iter(codes, batch_size=64)
y_pred = [p.target for p in preds]

report = classification_report(y_true, y_pred)
print(format_report(report))

with open('/content/d1_report.json', 'w') as f:
    json.dump(report.to_dict(), f, indent=2)
shutil.copy('/content/d1_report.json', f'{DRIVE_D1}/d1_report.json')
print('\n✅ Report saved to Drive: NT230/data/d1/d1_report.json')